# 🤖 Chatbot de Agendamiento — Agente LangGraph + RAG + Memoria (v2)

**Caso organizacional:** Centro de Resonancia y Scanner *MedImage* — agenda exámenes de resonancia magnética y TAC, responde preguntas frecuentes sobre preparación y procedimientos,
y bloquea feriados nacionales de Chile.

### Mejoras respecto a la v1 (alineadas a indicadores de evaluación)

| Indicador | Mejora implementada |
|-----------|---------------------|
| **IE1** | Herramientas formales del agente declaradas con `@tool` (LangChain Core) |
| **IE2** | Agente construido con **LangGraph `create_react_agent`** — ciclo ReAct automático (LangChain/LangGraph ≥ 0.3) |
| **IE3** | **Memoria corto plazo**: `MemorySaver` de LangGraph (historial persistente por `thread_id`). **Memoria largo plazo**: tabla `historial_conversacion` en SQLite — persiste entre sesiones. |
| **IE4** | RAG FAISS con documentos etiquetados por fuente (`interna`/`externa`) integrado como herramienta del agente. |
| **IE5** | Ciclo ReAct automático de LangGraph con `VERBOSE_AGENTE=True` — muestra herramientas invocadas y observaciones en tiempo real. |
| **IE6** | Celda de trazabilidad RAG + celda de demo de planificación como evidencia técnica documentada. |

**Stack:** Groq (`ChatGroq`) · Gemini (`GoogleGenerativeAIEmbeddings`) · LangChain · LangGraph · FAISS · SQLite

**API keys** → agrega `GOOGLE_API_KEY`, `LLM_API_KEY` y `LANGSMITH_API_KEY` en los Secrets de Colab (🔑).


## 1. Instalación de dependencias

⚠️ Después de ejecutar, **reinicia el entorno** (`Entorno de ejecución → Reiniciar sesión`) y continúa desde la celda 2.


In [ ]:
!pip install -q --upgrade groq google-generativeai langchain langchain-community \
    langchain-google-genai langchain-groq faiss-cpu numpy python-dateutil requests pandas \
    langgraph

import groq, langchain, langchain_groq, langgraph
from importlib.metadata import version as pkg_version
print("groq        :", groq.__version__)
print("langchain   :", langchain.__version__)
print("langchain-groq:", langchain_groq.__version__)
print("langgraph   :", pkg_version("langgraph"))

## 2. Configuración de API Keys y clientes LangChain

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY").strip()
os.environ["LLM_API_KEY"]    = userdata.get("LLM_API_KEY").strip()

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

EMBEDDING_MODEL = "models/gemini-embedding-001"
# Modelo con soporte de tool calling en Groq.
# Si tu clave accede a otro modelo, actualiza este nombre.
GROQ_MODEL = "openai/gpt-oss-120b"

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=os.environ["LLM_API_KEY"],
    temperature=0.3,
)

def groq_generate(prompt: str, system: str = None, temperature: float = 0.3) -> str:
    """Genera texto usando ChatGroq a través de LangChain."""
    modelo = llm if temperature == 0.3 else llm.bind(temperature=temperature)
    mensajes = []
    if system:
        mensajes.append(SystemMessage(content=system))
    mensajes.append(HumanMessage(content=prompt))
    return modelo.invoke(mensajes).content.strip()

print(f"✅ Clientes LangChain configurados.")
print(f"   LLM   : {GROQ_MODEL}")
print(f"   Embed : {EMBEDDING_MODEL}")

# ── LangSmith tracing ──────────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"]    = userdata.get("LANGSMITH_API_KEY").strip()
os.environ["LANGCHAIN_PROJECT"]    = "chatbot-resonancia-v2"
print("✅ LangSmith tracing activado — proyecto: chatbot-resonancia-v2")

## 3. Fuente interna de conocimiento — FAQ del centro de imágenes

In [ ]:
from langchain_core.documents import Document

faq_data = [
    {"pregunta": "¿Cuál es el horario de atención?",
     "respuesta": "Atendemos de lunes a viernes de 8:00 a 19:00 y sábados de 9:00 a 14:00."},
    {"pregunta": "¿Cómo puedo cancelar un examen agendado?",
     "respuesta": "Puedes cancelar escribiendo 'cancelar' en este chat indicando tu fecha, o llamando al +56 32 234 5678. Se solicita avisar con al menos 24 horas de anticipación."},
    {"pregunta": "¿Qué medios de pago aceptan?",
     "respuesta": "Aceptamos efectivo, tarjetas de débito/crédito, transferencia bancaria, Fonasa e Isapres convenio. Consulta previamente si tu Isapre tiene convenio con nosotros."},
    {"pregunta": "¿Cuánto dura una resonancia magnética?",
     "respuesta": "Una resonancia magnética dura entre 30 y 60 minutos dependiendo de la región a examinar. Con contraste endovenoso puede extenderse hasta 90 minutos."},
    {"pregunta": "¿Cuánto dura un TAC?",
     "respuesta": "Un TAC (tomografía axial computarizada) dura entre 15 y 30 minutos. Con contraste puede tardar hasta 45 minutos."},
    {"pregunta": "¿Necesito prepararme para la resonancia?",
     "respuesta": "Para la mayoría de las resonancias no se requiere preparación especial. Para resonancias abdominales o pélvicas se recomienda ayuno de 4 horas. Infórmanos si tienes implantes metálicos, marcapasos o claustrofobia."},
    {"pregunta": "¿Necesito preparación para el TAC con contraste?",
     "respuesta": "Para TAC con contraste IV se requiere ayuno de 4 horas y examen de creatinina reciente (menos de 3 meses). Infórmanos si eres diabético, tienes alergia al yodo o insuficiencia renal."},
    {"pregunta": "¿Necesito orden médica para agendar?",
     "respuesta": "Sí, para todos los exámenes se requiere orden médica vigente. Puedes traerla el día del examen o enviarla previamente por WhatsApp al +56 32 234 5678."},
    {"pregunta": "¿Dónde están ubicados?",
     "respuesta": "Estamos ubicados en Av. Colón 2383, Valparaíso. Hay estacionamiento disponible en la calle y locomoción cercana (metro Puerto, micros 601 y 612)."},
    {"pregunta": "¿Pueden personas con claustrofobia hacerse una resonancia?",
     "respuesta": "Sí. Contamos con equipo de resonancia de apertura amplia. En casos severos coordinamos con el médico tratante la administración de sedación leve. Infórmanos al momento de agendar."},
    {"pregunta": "¿Cuánto tiempo demoran los resultados?",
     "respuesta": "Los informes de resonancia y TAC están disponibles en 24 a 48 horas hábiles. Casos de urgencia se informan el mismo día. Los resultados se entregan en formato digital y físico."},
    {"pregunta": "¿Atienden urgencias?",
     "respuesta": "Atendemos urgencias según disponibilidad. Llama directamente al +56 32 234 5678 para coordinar un horario de urgencia fuera de la agenda habitual."},
]

faq_docs = [
    Document(
        page_content=f"Pregunta: {item['pregunta']}\nRespuesta: {item['respuesta']}",
        metadata={"fuente": "interna", "tipo": "faq",
                  "pregunta": item["pregunta"], "respuesta": item["respuesta"]},
    )
    for item in faq_data
]
print(f"✅ {len(faq_docs)} documentos internos (FAQ) cargados.")

## 4. Fuente externa — Feriados nacionales de Chile (API gov.cl)

La API pública `apis.digital.gob.cl/fl/feriados` no requiere API key.
Si no hay conexión, se usa el respaldo local.


In [ ]:
import requests
from datetime import date

FERIADOS_FALLBACK = [
    {"fecha": "2026-01-01", "nombre": "Año Nuevo",          "tipo": "Civil"},
    {"fecha": "2026-09-18", "nombre": "Fiestas Patrias",    "tipo": "Civil"},
    {"fecha": "2026-09-19", "nombre": "Glorias del Ejército","tipo": "Civil"},
    {"fecha": "2026-12-25", "nombre": "Navidad",             "tipo": "Religioso"},
]

def obtener_feriados_externos(anio: int = None):
    """Obtiene feriados desde la API pública del gobierno de Chile (fuente externa)."""
    anio = anio or date.today().year
    url  = f"https://apis.digital.gob.cl/fl/feriados/{anio}"
    try:
        resp = requests.get(url, timeout=8)
        resp.raise_for_status()
        datos = resp.json()
        feriados = [{"fecha": f["fecha"], "nombre": f["nombre"], "tipo": f.get("tipo", "")}
                    for f in datos]
        print(f"✅ Fuente externa OK: {len(feriados)} feriados obtenidos para {anio}.")
        return feriados
    except Exception as e:
        print(f"⚠️  API externa no disponible ({e}). Usando respaldo local.")
        return FERIADOS_FALLBACK

feriados_data = obtener_feriados_externos()

feriados_docs = [
    Document(
        page_content=(
            f"El {f['fecha']} es feriado nacional: {f['nombre']} ({f['tipo']}). "
            f"No se pueden agendar horas ese día."
        ),
        metadata={"fuente": "externa", "tipo": "feriado",
                  "fecha": f["fecha"], "nombre": f["nombre"]},
    )
    for f in feriados_data
]

feriados_fechas = {f["fecha"] for f in feriados_data}   # conjunto para lookups O(1)
print(f"✅ {len(feriados_docs)} documentos externos (feriados) cargados.")

## 5. Índice RAG combinado (LangChain + FAISS) — IE4

Un único índice vectorial mezcla **documentos internos** (FAQ) y **externos** (feriados).
Cada documento conserva su metadato `fuente`, lo que garantiza trazabilidad (IL1.4 / IE4).


In [ ]:
from langchain_community.vectorstores import FAISS

todos_los_docs = faq_docs + feriados_docs
vectorstore   = FAISS.from_documents(todos_los_docs, embeddings)
print(f"✅ Índice FAISS creado con {len(todos_los_docs)} documentos "
      f"({len(faq_docs)} internos + {len(feriados_docs)} externos).")

def buscar_contexto(pregunta: str, k: int = 3, umbral: float = 0.75):
    """
    Recupera los k documentos más relevantes (de ambas fuentes) para una pregunta.
    Devuelve lista de (Document, score). Score más bajo = más similar (distancia L2).
    Filtra resultados por debajo del umbral de similitud definido.
    """
    resultados  = vectorstore.similarity_search_with_score(pregunta, k=k)
    relevantes  = [(doc, score) for doc, score in resultados if score <= umbral]
    return relevantes

## 6. IE3 — Memoria de corto plazo (LangGraph `MemorySaver`)

LangGraph incluye un sistema de **checkpointing** que persiste automáticamente el historial
de mensajes de cada conversación identificada por un `thread_id`.

- **`MemorySaver`**: checkpointer en memoria (RAM) — persiste durante la sesión del notebook.
- El agente recibe el historial completo del `thread_id` en cada invocación: no se necesita
  pasar `chat_history` manualmente (a diferencia del ciclo manual anterior).
- Cambiar el `thread_id` inicia una conversación desde cero (simula múltiples usuarios/sesiones).

> **Comparativa con la v1:** LangChain ≥ 0.3 eliminó `ConversationBufferWindowMemory`.
> `MemorySaver` es la API moderna equivalente integrada nativamente en LangGraph.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# MemorySaver: checkpointer de corto plazo — gestiona historial por thread_id
# LangGraph lo consulta automáticamente en cada invoke() del agente
checkpointer_cp = MemorySaver()

print("✅ Memoria de corto plazo configurada.")
print("   Implementación : LangGraph MemorySaver")
print("   Mecanismo      : Historial de mensajes indexado por thread_id")
print("   Ventana        : Gestionada automáticamente por LangGraph")
print()
print("   → Al invocar agente.invoke(..., config={'configurable': {'thread_id': X}})")
print("     LangGraph recupera y almacena automáticamente el historial de la sesión X.")

## 7. IE3 — Memoria de largo plazo + Base de datos (SQLite)

SQLite almacena:
- **`citas`**: reservas de pacientes (agendadas / canceladas).
- **`historial_conversacion`**: todos los turnos de todas las sesiones — persiste
  entre reinicios del notebook y permite recuperar conversaciones anteriores.

Cada sesión recibe un `SESION_ID` único que permite filtrar su historial.


In [ ]:
import sqlite3, uuid, pandas as pd
from datetime import datetime

DB_PATH  = "citas.db"
conexion = sqlite3.connect(DB_PATH, check_same_thread=False)

# ── Tabla de citas ──────────────────────────────────────────────
conexion.execute("""
CREATE TABLE IF NOT EXISTS citas (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre     TEXT    NOT NULL,
    fecha_hora TEXT    NOT NULL,                    -- 'YYYY-MM-DD HH:MM'
    estado     TEXT    NOT NULL DEFAULT 'agendada', -- 'agendada' | 'cancelada'
    creado_en  TEXT    NOT NULL
);
""")

# ── Tabla de historial de conversación (memoria de largo plazo) ─
conexion.execute("""
CREATE TABLE IF NOT EXISTS historial_conversacion (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    sesion_id  TEXT    NOT NULL,
    rol        TEXT    NOT NULL,   -- 'usuario' | 'asistente'
    contenido  TEXT    NOT NULL,
    timestamp  TEXT    NOT NULL
);
""")
conexion.commit()

# ID único por ejecución del notebook
SESION_ID = str(uuid.uuid4())[:8]

def guardar_turno_lp(rol: str, contenido: str) -> None:
    """Persiste un turno en la memoria de largo plazo (SQLite)."""
    conexion.execute(
        "INSERT INTO historial_conversacion (sesion_id, rol, contenido, timestamp) "
        "VALUES (?, ?, ?, ?)",
        (SESION_ID, rol, contenido, datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    )
    conexion.commit()

def recuperar_historial_lp(sesion_id: str = None, limite: int = 20) -> list:
    """
    Recupera mensajes de la memoria de largo plazo para una sesión.
    sesion_id=None → sesión actual (SESION_ID).
    """
    sid    = sesion_id or SESION_ID
    cursor = conexion.execute(
        "SELECT rol, contenido, timestamp FROM historial_conversacion "
        "WHERE sesion_id = ? ORDER BY id ASC LIMIT ?",
        (sid, limite),
    )
    return [dict(zip(["rol", "contenido", "timestamp"], f)) for f in cursor.fetchall()]

def ver_sesiones_lp() -> pd.DataFrame:
    """Resumen de todas las sesiones guardadas en el historial de largo plazo."""
    return pd.read_sql_query(
        "SELECT sesion_id, COUNT(*) AS turnos, "
        "MIN(timestamp) AS inicio, MAX(timestamp) AS fin "
        "FROM historial_conversacion GROUP BY sesion_id ORDER BY inicio DESC",
        conexion,
    )

print(f"✅ Base de datos SQLite configurada: {DB_PATH}")
print(f"   Tablas    : citas, historial_conversacion")
print(f"   Sesión ID : {SESION_ID}  (memoria de largo plazo)")

## 8. Sistema de agendamiento — validación de horario y feriados


In [ ]:
import re
import dateutil.parser as dparser
from datetime import datetime, timedelta

HORARIO_APERTURA  = 8
HORARIO_CIERRE    = 19
DURACION_CITA_MIN = 60   # Resonancia: 60 min | TAC: 30 min (usamos 60 como bloque base)

PATRON_FECHA_HORA_ESTRICTA = re.compile(
    r"^\s*\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\s+\d{1,2}:\d{2}\s*$"
)
DIAS_SEMANA_ES = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]


def es_feriado(fecha_hora: datetime) -> bool:
    return fecha_hora.strftime("%Y-%m-%d") in feriados_fechas


def consultar_citas(nombre: str = None, fecha: str = None,
                    estado: str = "agendada") -> list:
    """Consulta la tabla de citas con filtros opcionales."""
    query  = "SELECT id, nombre, fecha_hora, estado, creado_en FROM citas WHERE 1=1"
    params = []
    if nombre:
        query += " AND LOWER(nombre) LIKE ?"
        params.append(f"%{nombre.lower()}%")
    if fecha:
        query += " AND fecha_hora LIKE ?"
        params.append(f"{fecha}%")
    if estado:
        query += " AND estado = ?"
        params.append(estado)
    query += " ORDER BY fecha_hora ASC"
    cursor  = conexion.execute(query, params)
    columnas = [c[0] for c in cursor.description]
    return [dict(zip(columnas, fila)) for fila in cursor.fetchall()]


def horario_disponible(fecha_hora: datetime) -> bool:
    if es_feriado(fecha_hora):
        return False
    if fecha_hora.hour < HORARIO_APERTURA or fecha_hora.hour >= HORARIO_CIERRE:
        return False
    for cita in consultar_citas(estado="agendada"):
        cita_dt = dparser.parse(cita["fecha_hora"])
        if abs((cita_dt - fecha_hora).total_seconds()) < DURACION_CITA_MIN * 60:
            return False
    return True


def normalizar_fecha_hora(texto: str) -> str:
    """Llama al LLM para convertir lenguaje natural a 'DD-MM-YYYY HH:MM'."""
    hoy = datetime.now()
    contexto_fecha = (
        f"{DIAS_SEMANA_ES[hoy.weekday()]} {hoy.strftime('%d-%m-%Y')}, "
        f"hora actual {hoy.strftime('%H:%M')}"
    )
    prompt = f"""Hoy es {contexto_fecha}.

Convierte la siguiente expresion de fecha y hora en espanol al formato EXACTO "DD-MM-YYYY HH:MM"
en 24 horas. Resuelve fechas relativas (manana, el lunes que viene, en dos dias) y horas en
palabras (4 de la tarde = 16:00, mediodia = 12:00, 9 de la manana = 09:00).
Si no hay fecha y hora determinable, responde exactamente: NO_ENCONTRADO

Expresion: "{texto}"
Responde SOLO con la fecha en el formato pedido (o NO_ENCONTRADO), sin texto adicional."""
    return groq_generate(prompt, temperature=0).strip()


def parsear_fecha_hora(texto: str):
    """Interpreta un texto de fecha/hora. Devuelve datetime o None."""
    texto = (texto or "").strip()
    if not texto:
        return None
    if PATRON_FECHA_HORA_ESTRICTA.match(texto):
        try:
            return dparser.parse(texto, dayfirst=True, fuzzy=True)
        except Exception:
            pass
    normalizado = normalizar_fecha_hora(texto)
    if normalizado == "NO_ENCONTRADO":
        return None
    try:
        return datetime.strptime(normalizado, "%d-%m-%Y %H:%M")
    except Exception:
        return None


def agendar_hora(nombre: str, fecha_hora_str: str) -> str:
    fecha_hora = parsear_fecha_hora(fecha_hora_str)
    if fecha_hora is None:
        return ("No logré entender la fecha/hora. "
                "Usa el formato 'DD-MM-YYYY HH:MM' o lenguaje natural.")
    if fecha_hora < datetime.now():
        return "Esa fecha ya pasó. Por favor indica una fecha futura."
    if es_feriado(fecha_hora):
        return (f"El {fecha_hora.strftime('%d-%m-%Y')} es feriado nacional "
                f"(fuente: API oficial de feriados de Chile). "
                f"No atendemos ese día. ¿Eliges otra fecha?")
    if not horario_disponible(fecha_hora):
        return (f"El horario {fecha_hora.strftime('%d-%m-%Y %H:%M')} no está disponible. "
                f"Elige dentro de {HORARIO_APERTURA}:00–{HORARIO_CIERRE}:00.")
    conexion.execute(
        "INSERT INTO citas (nombre, fecha_hora, estado, creado_en) VALUES (?, ?, 'agendada', ?)",
        (nombre, fecha_hora.strftime("%Y-%m-%d %H:%M"),
         datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    )
    conexion.commit()
    return (f"¡Listo, {nombre}! Examen agendado para el {fecha_hora.strftime('%d-%m-%Y %H:%M')}. "
            f"Recuerda traer tu orden médica vigente y llegar 10 minutos antes.")


def cancelar_hora(nombre: str, fecha_hora_str: str = None) -> str:
    citas_nombre = consultar_citas(nombre=nombre, estado="agendada")
    if not citas_nombre:
        return f"No encontré citas agendadas a nombre de {nombre}."
    if fecha_hora_str:
        fecha_hora = parsear_fecha_hora(fecha_hora_str)
        if fecha_hora is None:
            return "No entendí la fecha a cancelar."
        objetivo     = fecha_hora.strftime("%Y-%m-%d %H:%M")
        coincidencias = [c for c in citas_nombre if c["fecha_hora"] == objetivo]
        if not coincidencias:
            return f"No encontré cita de {nombre} para el {fecha_hora.strftime('%d-%m-%Y %H:%M')}."
        for c in coincidencias:
            conexion.execute("UPDATE citas SET estado = 'cancelada' WHERE id = ?", (c["id"],))
        conexion.commit()
        return f"Cita del {fecha_hora.strftime('%d-%m-%Y %H:%M')} cancelada para {nombre}."
    else:
        ids = [c["id"] for c in citas_nombre]
        conexion.executemany("UPDATE citas SET estado = 'cancelada' WHERE id = ?",
                             [(i,) for i in ids])
        conexion.commit()
        return f"Se cancelaron todas las citas de {nombre} ({len(ids)} en total)."


def ver_tabla_citas(estado: str = None) -> pd.DataFrame:
    """Muestra la tabla de citas como DataFrame (útil dentro del notebook)."""
    query  = "SELECT id, nombre, fecha_hora, estado, creado_en FROM citas"
    params = []
    if estado:
        query += " WHERE estado = ?"
        params.append(estado)
    query += " ORDER BY fecha_hora ASC"
    return pd.read_sql_query(query, conexion, params=params)


print("✅ Sistema de agendamiento configurado.")
print(f"   Horario  : L–V {HORARIO_APERTURA}:00–{HORARIO_CIERRE}:00 | Sáb 9:00–14:00")
print(f"   Bloque   : {DURACION_CITA_MIN} min por examen (resonancia / TAC)")

## 9. IE1 / IE2 — Herramientas formales del agente (`@tool`)

Cada capacidad del agente se declara como una herramienta LangChain con:
- **Nombre** semántico (el agente lo selecciona por nombre y descripción)
- **Firma tipada** (genera el JSON schema para el _tool calling_ del LLM)
- **Docstring** descriptivo (instrucción para el LLM sobre cuándo y cómo usarla)


In [ ]:
from langchain_core.tools import tool

@tool
def agendar_cita(nombre: str, fecha_hora: str) -> str:
    """
    Agenda un examen de imágenes médicas para un paciente en el Centro de Resonancia y Scanner MedImage.

    Úsala cuando el usuario quiera reservar, agendar o pedir un examen (resonancia, TAC u otro).
    Requiere el nombre completo del paciente y la fecha/hora deseada.

    Args:
        nombre     : Nombre completo del paciente (ej: 'María García').
        fecha_hora : Fecha y hora deseada en cualquier formato
                     (ej: '25 de octubre a las 10 de la mañana',
                          'mañana a las 15:30', '25-10-2026 10:00').

    Returns:
        Confirmación del agendamiento con instrucciones de preparación, o mensaje de error con alternativas.
    """
    return agendar_hora(nombre, fecha_hora)


@tool
def cancelar_cita(nombre: str, fecha_hora: str = "") -> str:
    """
    Cancela uno o todos los exámenes agendados de un paciente.

    Úsala cuando el usuario quiera cancelar un examen existente.
    Si no se especifica fecha/hora, se cancelan TODOS los exámenes del paciente.

    Args:
        nombre     : Nombre del paciente cuyo examen se quiere cancelar.
        fecha_hora : (Opcional) Fecha y hora específica a cancelar.
                     Dejar vacío para cancelar todos los exámenes del paciente.

    Returns:
        Confirmación de cancelación o mensaje si no se encontró el examen.
    """
    fh = fecha_hora.strip() if fecha_hora else None
    return cancelar_hora(nombre, fh)


@tool
def buscar_informacion_clinica(pregunta: str) -> str:
    """
    Busca información en la base de conocimiento del centro de imágenes y fuentes externas.

    Úsala para responder preguntas sobre: preparación para exámenes, duración de resonancias y TAC,
    requisitos (orden médica, creatinina, ayuno), horarios, medios de pago, ubicación,
    feriados nacionales de Chile y cualquier otro aspecto del servicio.
    Los documentos recuperados indican su fuente (interna/FAQ o externa/feriados).

    Args:
        pregunta : Pregunta del usuario sobre el servicio o los exámenes.

    Returns:
        Fragmentos relevantes con indicación de fuente (interna o externa).
    """
    contexto = buscar_contexto(pregunta, k=3)
    if not contexto:
        return "No se encontró información relevante en las fuentes disponibles."
    partes = []
    for doc, score in contexto:
        fuente = doc.metadata.get("fuente", "desconocida")
        partes.append(f"[Fuente {fuente} | similitud {score:.3f}] {doc.page_content}")
    return "\n\n".join(partes)


@tool
def verificar_disponibilidad(fecha_hora: str) -> str:
    """
    Verifica si una fecha y hora está disponible para agendar una cita.

    Considera: feriados nacionales, horario de atención y citas ya existentes.
    Úsala antes de agendar si el usuario pregunta por disponibilidad,
    o para diagnosticar por qué un horario no está disponible.

    Args:
        fecha_hora : Fecha y hora a verificar en cualquier formato
                     (ej: '25-10-2026 10:00', '25 de octubre a las 10',
                          'mañana a las 15:30').

    Returns:
        Estado de disponibilidad con motivo detallado si no está disponible.
    """
    dt = parsear_fecha_hora(fecha_hora)
    if dt is None:
        return "No se pudo interpretar la fecha/hora. Intenta con formato 'DD-MM-YYYY HH:MM'."
    if dt < datetime.now():
        return f"La fecha {dt.strftime('%d-%m-%Y %H:%M')} ya pasó."
    if es_feriado(dt):
        return f"❌ El {dt.strftime('%d-%m-%Y')} es feriado nacional (no atendemos ese día)."
    if dt.weekday() == 6:
        return f"❌ El {dt.strftime('%d-%m-%Y')} es domingo (no atendemos)."
    if dt.weekday() == 5 and dt.hour >= 13:
        return "❌ Los sábados solo atendemos hasta las 13:00."
    if dt.hour < HORARIO_APERTURA or dt.hour >= HORARIO_CIERRE:
        return (f"❌ Las {dt.strftime('%H:%M')} está fuera del horario de atención "
                f"({HORARIO_APERTURA}:00–{HORARIO_CIERRE}:00).")
    if horario_disponible(dt):
        return f"✅ El {dt.strftime('%d-%m-%Y a las %H:%M')} está disponible."
    return (f"❌ El {dt.strftime('%d-%m-%Y %H:%M')} ya está ocupado. "
            f"Elige otro horario (citas cada {DURACION_CITA_MIN} min).")


@tool
def listar_citas_paciente(nombre: str) -> str:
    """
    Consulta los exámenes actualmente agendados de un paciente por nombre.

    Úsala para informar al usuario sobre sus reservas, o antes de cancelar
    cuando hay múltiples exámenes y se necesita confirmar cuál cancelar.

    Args:
        nombre : Nombre del paciente (búsqueda parcial, sin distinguir mayúsculas).

    Returns:
        Lista de exámenes agendados con fecha/hora e ID, o mensaje si no hay exámenes.
    """
    citas = consultar_citas(nombre=nombre, estado="agendada")
    if not citas:
        return f"No hay exámenes agendados a nombre de '{nombre}'."
    lineas = [f"Exámenes agendados para '{nombre}':"]
    for c in citas:
        lineas.append(f"  • {c['fecha_hora']}  (ID #{c['id']})")
    return "\n".join(lineas)


herramientas = [
    agendar_cita,
    cancelar_cita,
    buscar_informacion_clinica,
    verificar_disponibilidad,
    listar_citas_paciente,
]

print(f"✅ {len(herramientas)} herramientas del agente registradas (IE1/IE2):")
for h in herramientas:
    desc_corta = h.description.strip().split("\n")[0][:70]
    print(f"   • {h.name:<30}  {desc_corta}...")

## 10. IE2 / IE5 — Agente con planificación (`create_react_agent` de LangGraph)

LangChain 0.3+ reemplaza `AgentExecutor` con **LangGraph** como framework oficial de agentes.
`create_react_agent` de `langgraph.prebuilt` implementa el ciclo **ReAct** automáticamente:

| Paso | Descripción |
|------|-------------|
| **Reason** | El LLM recibe el sistema, el historial (vía `MemorySaver`) y la consulta actual. Razona y decide qué herramienta usar (o responder directamente). |
| **Act** | Si el LLM emite `tool_calls`, LangGraph invoca las herramientas en paralelo o secuencia según el modelo. |
| **Observe** | El resultado de cada herramienta se añade como `ToolMessage` al contexto del grafo. |
| **Repeat / Respond** | El ciclo repite hasta que el LLM genera una respuesta final sin `tool_calls`. |

**Ventajas vs. ciclo manual anterior:**
- El ciclo ReAct está encapsulado → código más limpio y mantenible.
- `MemorySaver` gestiona el historial completo por `thread_id` sin código extra.
- LangSmith traza el **grafo completo** de LangGraph (nodos, herramientas, mensajes).
- Compatible con streaming (`agente.stream()`), interrupciones humanas y checkpoints.


In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

SYSTEM_PROMPT_AGENTE = """Eres el asistente virtual del Centro de Resonancia y Scanner MedImage, ubicado en Av. Colón 2383, Valparaíso.
Tu función es gestionar el agendamiento de exámenes de imágenes médicas (resonancia magnética y TAC) y responder preguntas sobre procedimientos, preparación y el servicio.

Capacidades:
- Agendar exámenes para pacientes
- Cancelar exámenes existentes
- Verificar disponibilidad de horarios
- Informar sobre preparación, duración y requisitos de cada examen

Reglas de negocio:
- Horario: Lunes a Viernes 8:00-19:00, Sábados 9:00-14:00
- No atendemos en feriados nacionales chilenos
- Bloque base por examen: 60 minutos
- Todos los exámenes requieren orden médica vigente
- Para exámenes con contraste IV se requiere ayuno de 4 horas y creatinina reciente

Instrucciones de comportamiento:
- USA SIEMPRE las herramientas para obtener datos precisos; nunca inventes información
- Si falta el nombre del paciente para agendar/cancelar, pregúntalo antes de actuar
- Si falta la fecha/hora o el tipo de examen, pregúntalo antes de usar la herramienta
- Mantén un tono amable, profesional y empático (los pacientes pueden estar nerviosos)
- Cuando el horario no esté disponible, sugiere alternativas cercanas
- Si el paciente menciona claustrofobia o implantes metálicos, recuérdale informarlo al momento de agendar"""

# ── Construcción del agente LangGraph ──────────────────────────
# create_react_agent:
#   model       → LLM (vincula herramientas internamente vía bind_tools)
#   tools       → lista de herramientas @tool
#   prompt      → system prompt (str o callable)
#   checkpointer→ MemorySaver para memoria de corto plazo (IE3)
agente = create_react_agent(
    model=llm,
    tools=herramientas,
    prompt=SYSTEM_PROMPT_AGENTE,
    checkpointer=checkpointer_cp,
)

# Configuración de sesión: thread_id identifica la conversación activa
# Cambiar thread_id inicia una nueva conversación desde cero
config_agente = {"configurable": {"thread_id": SESION_ID}}

VERBOSE_AGENTE = True  # Muestra herramientas invocadas en el loop interactivo (IE5)

print("✅ Agente LangGraph configurado (IE2 / IE5).")
print(f"   Framework    : LangGraph create_react_agent (langgraph.prebuilt)")
print(f"   Ciclo        : ReAct automático — Reason → Act → Observe → Respond")
print(f"   Memoria CP   : MemorySaver (thread_id = {SESION_ID})")
print(f"   LangSmith    : trazado automático del grafo completo")
print(f"   Herramientas : {[h.name for h in herramientas]}")

## 11. Prueba de trazabilidad RAG — IE4 / IE6

Muestra explícitamente qué documentos y de qué fuente se recuperan para distintas preguntas.
Sirve como evidencia técnica del sistema RAG para el informe (IE6).


In [ ]:
print("=== Trazabilidad RAG — IE4 / IE6 ===\n")

preguntas_prueba = [
    "¿puedo ir el 18 de septiembre?",
    "¿cómo cancelo mi examen?",
    "¿necesito ayuno para la resonancia?",
    "¿cuánto dura un TAC con contraste?",
    "¿necesito orden médica?",
]

for p in preguntas_prueba:
    print(f"Pregunta: {p}")
    resultados = buscar_contexto(p, k=2)
    if not resultados:
        print("  (sin resultados por encima del umbral de similitud)")
    for doc, score in resultados:
        fuente = doc.metadata.get("fuente", "?")
        print(f"  → fuente={fuente} | score={score:.3f} | {doc.page_content[:80]}...")
    print()

## 12. Demo de planificación adaptativa — IE5

Esta celda ejecuta el agente en dos escenarios distintos y muestra:
1. Los **pasos intermedios**: qué herramientas invocó el agente y sus resultados.
2. La **respuesta final** generada tras razonar sobre las observaciones.

Se inspeccionan los mensajes del resultado (`resultado["messages"]`) para reconstruir
el ciclo Reason → Act → Observe que ejecutó LangGraph internamente.


In [ ]:
print("=" * 65)
print("DEMO IE5 — Planificación y Toma de Decisiones del Agente")
print("=" * 65)

# Usamos un thread separado para el demo para no contaminar la sesión principal
config_demo = {"configurable": {"thread_id": f"{SESION_ID}-demo"}}

def ejecutar_demo(consulta: str, label: str) -> str:
    """Ejecuta el agente y muestra los pasos de planificación (IE5)."""
    print(f"\n📌 {label}")
    print(f"   Consulta: {consulta}")
    print("-" * 65)

    resultado = agente.invoke(
        {"messages": [HumanMessage(content=consulta)]},
        config=config_demo,
    )

    # Reconstruir pasos intermedios desde los mensajes del grafo
    print("\n📋 Pasos de planificación ejecutados:")
    paso = 0
    for msg in resultado["messages"]:
        # AIMessage con tool_calls → el agente decidió usar una herramienta
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                paso += 1
                args_str = str(tc["args"])[:80]
                print(f"  Paso {paso}: [Reason→Act] herramienta={tc['name']}  args={args_str}")
        # ToolMessage → resultado de la herramienta (Observe)
        elif isinstance(msg, ToolMessage):
            obs_str = str(msg.content)[:120]
            print(f"          [Observe]     resultado={obs_str}")

    respuesta_final = resultado["messages"][-1].content
    print(f"\n✅ Respuesta final:\n{respuesta_final}")
    return respuesta_final

# ── Escenario A: Agendar en feriado ─────────────────────────────
ejecutar_demo(
    "Necesito una resonancia para el 1 de enero a las 10:00. Me llamo Carlos Vega.",
    "Escenario A: intento de agendar examen en feriado nacional"
)

print("\n" + "=" * 65)

# ── Escenario B: Pregunta sobre preparación ──────────────────────
ejecutar_demo(
    "¿Necesito prepararme para una resonancia con contraste? ¿Cuánto dura?",
    "Escenario B: consulta sobre preparación para examen con contraste"
)

## 13. Loop de chat interactivo

Comandos especiales disponibles:
- `historial` — muestra la memoria de **largo plazo** de la sesión actual (SQLite).
- `sesiones`  — resumen de todas las sesiones guardadas en SQLite.
- `citas`     — tabla de citas agendadas/canceladas.
- `memoria`   — muestra el historial de **corto plazo** desde LangGraph `MemorySaver`.
- `salir`     — termina el chat.


In [ ]:
print(f"🏥 Asistente de agendamiento MedImage listo. Sesión: {SESION_ID}")
print("   Escribe 'salir' para terminar o un comando especial (historial, sesiones, citas, memoria)\n")

while True:
    try:
        entrada = input("Tu: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n🤖 ¡Hasta luego!")
        break

    if not entrada:
        continue

    if entrada.lower() in ["salir", "exit", "quit"]:
        print("🤖 ¡Hasta luego!")
        break

    # ── Comandos de inspección ───────────────────────────────────
    if entrada.lower() == "historial":
        hist = recuperar_historial_lp()
        print(f"\n📚 Historial largo plazo — sesión {SESION_ID} ({len(hist)} mensajes):")
        for msg in hist:
            print(f"  [{msg['timestamp']}] {msg['rol'].upper()}: {msg['contenido'][:90]}...")
        print()
        continue

    if entrada.lower() == "sesiones":
        print("\n📊 Sesiones en memoria de largo plazo:")
        print(ver_sesiones_lp().to_string(index=False))
        print()
        continue

    if entrada.lower() == "citas":
        print("\n📅 Tabla de citas:")
        print(ver_tabla_citas().to_string(index=False))
        print()
        continue

    if entrada.lower() == "memoria":
        # Inspeccionar historial de corto plazo desde LangGraph MemorySaver
        estado = agente.get_state(config_agente)
        msgs   = estado.values.get("messages", [])
        print(f"\n🧠 Memoria de corto plazo — MemorySaver ({len(msgs)} mensajes | thread='{SESION_ID}'):")
        for m in msgs:
            tipo = type(m).__name__.replace("Message", "")
            contenido = str(m.content) if isinstance(m.content, str) else str(m.content)[:90]
            print(f"  [{tipo:10}] {contenido[:90]}...")
        print()
        continue

    # ── Turno normal del agente ──────────────────────────────────
    guardar_turno_lp("usuario", entrada)

    try:
        resultado = agente.invoke(
            {"messages": [HumanMessage(content=entrada)]},
            config=config_agente,
        )
        respuesta = resultado["messages"][-1].content

        # Verbose: mostrar herramientas invocadas en este turno (IE5)
        if VERBOSE_AGENTE:
            for msg in resultado["messages"]:
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"  🔧 [{tc['name']}] {str(tc['args'])[:80]}")

    except Exception as e:
        respuesta = f"Lo siento, ocurrió un error al procesar tu solicitud. ({e})"

    guardar_turno_lp("asistente", respuesta)
    print(f"🤖 Bot: {respuesta}\n")